In [12]:
import os
import sys
import torch
from pathlib import Path
from tqdm import tqdm

ROOT_DIR = "/root/private_data/luog/codex/IgGM2"
os.chdir(ROOT_DIR)
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"当前工作目录已切换为: {os.getcwd()}")

from src.iggm_lightning.data_module import ProcessedSabdabDataModule
from IgGM.model.arch.core.diffuser import Diffuser
# 新增引入提取 local coords 的工具函数
from IgGM.utils import extract_per_loop_clean_local_coords

def calculate_translation_and_local_stats():
    # 数据路径配置
    metadata_path = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab/metadata.json"
    pdb_dir = "/root/private_data/luog/codex/IgGM2/data/sabdab/pdb"
    samples_dir = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab/samples"
    train_ids_path = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab_file/split_debug_one/train_prot_ids.txt"
    # train_ids_path = "/root/private_data/luog/codex/IgGM2/data/sabdab/sabdab_file/split_debug/test_prot_ids.txt"

    print("初始化 DataModule ...")
    
    datamodule = ProcessedSabdabDataModule(
        metadata_path=metadata_path,
        pdb_dir=pdb_dir,
        train_ids_path=train_ids_path,
        samples_dir=samples_dir,
        batch_size=1, 
        num_workers=0,                
        n_steps=200,                  
        forward_chunk_size=128,       
        max_antigen_len=256           
    )
    datamodule.setup(stage="fit")
    
    train_loader = torch.utils.data.DataLoader(
        datamodule.train_ds, 
        batch_size=1, 
        shuffle=False, 
        num_workers=0, 
        collate_fn=lambda x: x[0]
    )

    all_trsl = []
    all_local_coords = [] # 用于收集有效的 CDR local 坐标

    print("开始遍历训练集提取特征 ...")
    device = torch.device("cpu")
    
    for item in tqdm(train_loader):
        try:
            prot_data = item["payload"]["prot_data_curr"]
            
            # --- 1. 准备全局 FR 平移所需数据 ---
            cord_tns_orig = prot_data["cords_atom14"].to(device)

            # --- 2. 准备 CDR Local 坐标所需数据 ---
            loop_global_res_indices = prot_data["loop_global_res_indices"].to(device)
            loop_true_len = prot_data["loop_true_len"].to(device)
            loop_left_anchor_idx = prot_data["loop_left_anchor_idx"].to(device)
            loop_right_anchor_idx = prot_data["loop_right_anchor_idx"].to(device)
            loop_valid_res_mask = prot_data["loop_valid_res_mask"].to(device, dtype=torch.bool)
            loop_atom_valid_mask = prot_data["loop_atom_valid_mask"].to(device, dtype=torch.bool)
            
            # 构建 loop 真实原子 mask
            loop_atom_supervise_mask = loop_valid_res_mask.unsqueeze(-1).expand_as(loop_atom_valid_mask)

            # 提取干净的 loop 局部坐标
            clean_loop_local_coords, _, _ = extract_per_loop_clean_local_coords(
                cord_tns_orig, 
                loop_global_res_indices,
                loop_true_len,
                loop_left_anchor_idx,
                loop_right_anchor_idx,
                loop_atom_supervise_mask,
            )
            
            # 【核心过滤】只保留 mask 为 True 的有效原子坐标！
            valid_local_coords = clean_loop_local_coords[loop_atom_supervise_mask]

            
            # 将有效的局部坐标加入列表（避免全 0 填补影响方差计算）
            if valid_local_coords.numel() > 0:
                all_local_coords.append(valid_local_coords.clone().detach())

        except Exception as e:
            print(f"\n警告：跳过样本 {item.get('prot_id')}, 原因: {e}")
            continue


    # return clean_loop_local_coords, valid_local_coords
    
    # --- 统计 CDR Local 坐标参数 ---
    # 由于每个样本有效原子数量不同，这里使用 cat 而不是 stack拼接
    all_local_coords_tensor = torch.cat(all_local_coords, dim=0).float()
    cdr_local_mu = all_local_coords_tensor.mean(dim=0)
    cdr_local_scale_scalar = all_local_coords_tensor.std()

    print("\n" + "="*65)
    print("✅ 计算完成！请将以下参数直接替换到 diffuser.py 的 __init__ 方法中：")
    print("-" * 65)
    print("        # CDR 局部坐标的归一化参数")
    print(f"        self.cdr_local_mu = torch.tensor([{cdr_local_mu[0]:.4f}, {cdr_local_mu[1]:.4f}, {cdr_local_mu[2]:.4f}], dtype=torch.float32)")
    print(f"        self.cdr_local_scale = torch.tensor({cdr_local_scale_scalar:.4f}, dtype=torch.float32)")
    print("=" * 65)

if __name__ == "__main__":
    clean_loop_local_coords, valid_local_coords =   calculate_translation_and_local_stats()

当前工作目录已切换为: /root/private_data/luog/codex/IgGM2
初始化 DataModule ...
[DataModule] test split: all=1
开始遍历训练集提取特征 ...


100%|██████████| 99/99 [00:53<00:00,  1.85it/s]


✅ 计算完成！请将以下参数直接替换到 diffuser.py 的 __init__ 方法中：
-----------------------------------------------------------------
        # CDR 局部坐标的归一化参数
        self.cdr_local_mu = torch.tensor([-1.8734, 1.2467, -0.2309], dtype=torch.float32)
        self.cdr_local_scale = torch.tensor(5.8927, dtype=torch.float32)


TypeError: cannot unpack non-iterable NoneType object

In [13]:
clean_loop_local_coords.std()

tensor(4.5232)